In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import src.config_loader as config_loader
import util.read_price as read_price
import src.ip.optimizations as optimizations

print("config_loader:", config_loader.__file__)
print("read_price:", read_price.__file__)
print("rolling_ce:", optimizations.__file__)


battery, markets = config_loader.load_config(PROJECT_ROOT / "configs" / "battery_config.yaml")
ip_cfg = markets["ip"]

ip_det_xgb  = read_price.read_ip_forecast(model="XGB",  kind="deterministic", freq="15min")
ip_det_lear = read_price.read_ip_forecast(model="LEAR", kind="deterministic", freq="15min")
ip_det_qr   = read_price.read_ip_forecast(model="QR",   kind="deterministic", freq="15min")

ip_real = read_price.read_ip_real_prices(freq="15min", keep_extra_columns=False)["Price"]

start_date = "2023-01-01"
end_date = "2023-12-31"




config_loader: c:\Users\mmascare\OneDrive - KU Leuven\Documents\Code\DA_Optimization\src\config_loader.py
read_price: c:\Users\mmascare\OneDrive - KU Leuven\Documents\Code\DA_Optimization\util\read_price.py
rolling_ce: c:\Users\mmascare\OneDrive - KU Leuven\Documents\Code\DA_Optimization\src\ip\optimizations.py
C:\Users\mmascare\OneDrive - KU Leuven\Documents\Code\DA_Optimization\Data\IP_CET\IP_XGB.csv
C:\Users\mmascare\OneDrive - KU Leuven\Documents\Code\DA_Optimization\Data\IP_CET\IP_LEAR.csv
C:\Users\mmascare\OneDrive - KU Leuven\Documents\Code\DA_Optimization\Data\IP_CET\IP_QR.csv
C:\Users\mmascare\OneDrive - KU Leuven\Documents\Code\DA_Optimization\Data\IP_CET\IP_Real_Prices.csv


In [ ]:
# --- grids (from earlier suggestion) ---
terminal_penalty_L1 = [0.001, 0.003, 0.01, 0.03]          # €/kWh, L1
terminal_penalty_L2 = [1e-5, 3e-5, 1e-4, 3e-4]            # L2 scale (small!)
cycle_penalty_grid  = [0.0, 2.0, 5.0, 10.0, 20.0, 40.0]   # €/MWh throughput

# choose a fixed L1 terminal penalty for the cycle sweep
fixed_L1_terminal_penalty = 0.01  # pick from terminal_penalty_L1 (or whatever you want)

# common call kwargs
common_kwargs = dict(
    battery=battery,
    market=ip_cfg,
    forecasts={"qr": ip_det_qr},  # add other models if you want
    real_price_series=ip_real,
    price_source=["forecast"],#, "naive", "perfect_foresight"],
    start=start_date,
    end=end_date,
    terminal_target_kwh=battery.energy_kwh * 0.5,
    solver_name="gurobi_direct",
    save=True,
   #tag=None,  # set per run
)

results = {}  # store returned IPRollingResult objects if you want

# ---------------------------------------------------------------------
# (A) L1 sweep at cycle_penalty = 0
# ---------------------------------------------------------------------
for tp in terminal_penalty_L1:
    tag = f"CE_L1_tp{tp}".replace(".", "p")
    res = optimizations.run_ip_rolling_ce_models(
        **common_kwargs,
        terminal_penalty=float(tp),
        terminal_penalty_mode="L1",
        cycle_penalty_eur_per_mwh=0.0,
        #tag=tag,
    )
    results[f"L1_tp={tp}_cyc=0"] = res

# ---------------------------------------------------------------------
# (B) L2 sweep at cycle_penalty = 0
# ---------------------------------------------------------------------
for tp in terminal_penalty_L2:
    tag = f"CE_L2_tp{tp}".replace(".", "p").replace("-", "m")
    res = optimizations.run_ip_rolling_ce_models(
        **common_kwargs,
        terminal_penalty=float(tp),
        terminal_penalty_mode="L2",
        cycle_penalty_eur_per_mwh=0.0,
        #tag=tag,
    )
    results[f"L2_tp={tp}_cyc=0"] = res

# ---------------------------------------------------------------------
# (C) Cycle penalty sweep with fixed L1 terminal penalty
# ---------------------------------------------------------------------
for cyc in cycle_penalty_grid:
    tag = f"CE_L1_tp{fixed_L1_terminal_penalty}_cyc{cyc}".replace(".", "p")
    res = optimizations.run_ip_rolling_ce_models(
        **common_kwargs,
        terminal_penalty=float(fixed_L1_terminal_penalty),
        terminal_penalty_mode="L1",
        cycle_penalty_eur_per_mwh=float(cyc),
        #tag=tag,
    )
    results[f"L1_tp={fixed_L1_terminal_penalty}_cyc={cyc}"] = res

# Example: grab one history df
# df = results["L1_tp=0.01_cyc=0"].history


Solving forecast:  38.6 %